In [1]:
# 설치
!pip install transformers==4.45.0 -q
!pip install datasets -q
!pip install scikit-learn -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 71.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 58.3 MB/s eta 0:00:00:00:01


In [3]:
import json
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import DistilBertModel, DistilBertTokenizerFast
from datasets import load_dataset
from tqdm.auto import tqdm
from sklearn.linear_model import LinearRegression


# ─── 1. Dual-Stream Prototype Manifold ────────────────────────────────────────
class PrototypeManifold(nn.Module):
    def __init__(self, embed_dim: int, n_prototypes: int = 32):
        super().__init__()
        half = n_prototypes // 2
        self.proto_s = nn.Parameter(torch.randn(half, embed_dim))
        self.proto_c = nn.Parameter(torch.randn(half, embed_dim))

    def _stream_stats(self, z_norm, proto):
        p_norm = F.normalize(proto, dim=-1)
        sim = z_norm @ p_norm.T
        max_sim, _ = sim.max(dim=-1)
        return max_sim

    def forward(self, z: torch.Tensor) -> dict:
        z_norm = F.normalize(z, dim=-1)

        support = self._stream_stats(z_norm, self.proto_s)
        counter = self._stream_stats(z_norm, self.proto_c)

        # ── prototype collapse 방지 ──────────────────────────────────────
        s_norm = F.normalize(self.proto_s, dim=-1)
        c_norm = F.normalize(self.proto_c, dim=-1)

        cross_div = (s_norm @ c_norm.T).abs().mean()

        K_s = s_norm.size(0)
        K_c = c_norm.size(0)
        gram_s = s_norm @ s_norm.T
        gram_c = c_norm @ c_norm.T
        eye_s = torch.eye(K_s, device=z.device)
        eye_c = torch.eye(K_c, device=z.device)
        intra_s = (gram_s - eye_s).pow(2).mean()
        intra_c = (gram_c - eye_c).pow(2).mean()

        diversity_loss = cross_div + 0.5 * (intra_s + intra_c)

        return {
            "support": support,
            "counter": counter,
            "diversity_loss": diversity_loss,
        }


# ─── 2. Epistemic Field Classifier ────────────────────────────────────────────
class EpistemicFieldClassifier(nn.Module):

    AXES = ['truth', 'error', 'contradiction', 'novelty', 'ambiguity', 'ignorance']

    def __init__(self, eps: float = 1e-8):
        super().__init__()
        self.eps = eps
        self.scales = nn.Parameter(torch.ones(6))
        self.truth_temp = nn.Parameter(torch.tensor(0.2))

        self.contra_bias = nn.Parameter(torch.tensor(1.0))
        self.contra_temp = nn.Parameter(torch.tensor(0.2))

    def forward(self, manifold_out: dict, ignorance: torch.Tensor) -> dict:
        support = manifold_out["support"]
        counter = manifold_out["counter"]
        novelty_score = manifold_out["novelty_score"]
        ambiguity_score = manifold_out["ambiguity_score"]

        temp = F.softplus(self.truth_temp).clamp(min=0.05)
        margin_s = (support - counter).clamp(min=0.0)
        margin_c = (counter - support).clamp(min=0.0)

        # ── evidence plane ───────────────────────────────────────────────
        truth = support * torch.sigmoid(margin_s / temp)
        error = counter * torch.sigmoid(margin_c / temp)

        energy_sc = support + counter
        agree = (support - counter).abs().clamp(0.0, 1.0)
        c_temp = F.softplus(self.contra_temp).clamp(min=0.05)
        contradiction = torch.sigmoid((energy_sc - F.softplus(self.contra_bias)) / c_temp) * (1.0 - agree)

        # ── independent uncertainty sources ──────────────────────────────
        novelty = novelty_score
        ambiguity = ambiguity_score

        raw = torch.stack(
            [truth, error, contradiction, novelty, ambiguity, ignorance], dim=-1
        )
        scales_norm = F.softmax(self.scales, dim=0) * 6.0
        field = raw * scales_norm

        if self.training:
            dominant_type = None
            active_states = None
        else:
            dominant_idx = field.argmax(dim=-1)
            dominant_type = [self.AXES[i] for i in dominant_idx.cpu().tolist()]
            active_states = []
            for sample in field:
                th = sample.mean()
                active = [axis for idx, axis in enumerate(self.AXES) if sample[idx] > th]
                active_states.append(active if active else ["ignorance"])

        return {
            "field": field,
            "dominant_type": dominant_type,
            "active_states": active_states,
            "truth": truth,
            "error": error,
            "contradiction": contradiction,
            "novelty": novelty,
            "ambiguity": ambiguity,
            "ignorance": ignorance,
            "support_raw": support,
            "counter_raw": counter,
        }


# ─── Token-Novelty Source ─────────────────────────────────────────────────────
class TokenNovelty(nn.Module):
    """학습 어휘 대비 입력의 신규성 — 표면 형태, task/z와 완전 독립."""

    def __init__(self, vocab_size: int, eps: float = 1e-4):
        super().__init__()
        self.eps = eps
        self.register_buffer("token_count", torch.zeros(vocab_size))
        self.register_buffer("total", torch.tensor(0.0))
        # 특수토큰 마스킹용 (PAD/CLS/SEP은 신규성 계산서 제외)
        self.register_buffer("special_mask", torch.zeros(vocab_size, dtype=torch.bool))
        self.detach_out = True

    def set_special_tokens(self, ids):
        self.special_mask[torch.tensor(ids)] = True

    @torch.no_grad()
    def _update(self, input_ids, attention_mask):
        valid = input_ids[attention_mask.bool()]
        self.token_count.index_add_(0, valid, torch.ones_like(valid, dtype=torch.float))
        self.total += valid.numel()

    def forward(self, input_ids, attention_mask):
        if self.training:
            self._update(input_ids, attention_mask)

        # 토큰별 학습빈도 → 희귀도 = -log(freq), 미등장이면 최대
        total = self.total.clamp(min=1.0)
        freq = self.token_count[input_ids] / total          # (B, T)
        rarity = -(freq + self.eps).log()                    # 희귀할수록 큼

        # 특수토큰·패딩 제외하고 문장 평균
        mask = attention_mask.bool() & ~self.special_mask[input_ids]
        rarity = rarity * mask.float()
        denom = mask.float().sum(dim=-1).clamp(min=1.0)
        sent_rarity = rarity.sum(dim=-1) / denom             # (B,)

        # 즉석 정규화 — log(eps) 기준 (미등장 토큰 = -log(eps) 부근)
        max_rarity = -torch.log(torch.tensor(self.eps, device=input_ids.device))
        novelty = (sent_rarity / max_rarity).clamp(0.0, 1.0)
        return novelty.detach() if self.detach_out else novelty


# ─── Attention-based Ignorance Source ─────────────────────────────────────────
class AttentionIgnorance(nn.Module):
    """claim→evidence cross-attention 정렬 부족 = ignorance.
    mode='dispersion'은 ablation용 (방향성 없는 attention entropy)."""

    def __init__(self, sep_id: int = 102, momentum: float = 0.01, eps: float = 1e-6):
        super().__init__()
        self.sep_id = sep_id
        self.eps = eps
        self.momentum = momentum
        self.register_buffer("mis_mean", torch.tensor(0.5))
        self.register_buffer("mis_std", torch.tensor(0.15))
        self.register_buffer("initialized", torch.tensor(False))
        self.mode = "crossattn"      # "crossattn"(기본) | "dispersion"(ablation)
        self.detach_out = True

    @torch.no_grad()
    def _update(self, d):
        m = self.momentum
        if not self.initialized:
            self.mis_mean.copy_(d.mean())
            self.mis_std.copy_(d.std() + self.eps)
            self.initialized.fill_(True)
        else:
            self.mis_mean.mul_(1 - m).add_(d.mean(), alpha=m)
            self.mis_std.mul_(1 - m).add_(d.std() + self.eps, alpha=m)

    def forward(self, attentions, input_ids, attention_mask):
        A = attentions[-1].detach().mean(dim=1)               # (B, S, S) head 평균

        if self.mode == "dispersion":
            # ablation: cross-attention 대신 단순 attention entropy (방향성 없음)
            amask = attention_mask.bool()
            a = A * amask.unsqueeze(1).float()
            a = a / (a.sum(dim=-1, keepdim=True) + self.eps)
            ent = -(a * (a + self.eps).log()).sum(dim=-1)      # (B,S) 각 query entropy
            qf = amask.float()
            mis = (ent * qf).sum(-1) / qf.sum(-1).clamp(min=1.0)  # 길이평균 entropy
        else:
            # 기본: claim→evidence 정렬 (cross-attention)
            is_sep = (input_ids == self.sep_id)
            sep_cumsum = is_sep.cumsum(dim=1)
            amask = attention_mask.bool()
            claim_mask = (sep_cumsum == 0) & amask
            claim_mask[:, 0] = False
            evid_mask = (sep_cumsum == 1) & (~is_sep) & amask
            evid_f = evid_mask.unsqueeze(1).float()
            mass = (A * evid_f).sum(dim=-1)
            claim_f = claim_mask.float()
            align = (mass * claim_f).sum(dim=-1) / claim_f.sum(dim=-1).clamp(min=1.0)
            mis = 1.0 - align

        if self.training:
            self._update(mis.detach())
        mu = self.mis_mean.detach()
        std = self.mis_std.detach().clamp(min=self.eps)
        ignorance = torch.sigmoid((mis - mu) / std)
        return ignorance.detach() if self.detach_out else ignorance


# ─── Layer-Disagreement Ambiguity Source ──────────────────────────────────────
class LayerAmbiguity(nn.Module):
    """layer 간 [CLS] 방향 불일치 — 해석 비수렴. manifold(분류)와 독립 소스."""

    def __init__(self, momentum: float = 0.01, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.momentum = momentum
        self.register_buffer("disp_mean", torch.tensor(0.3))
        self.register_buffer("disp_std", torch.tensor(0.15))
        self.register_buffer("initialized", torch.tensor(False))
        self.detach_out = True

    @torch.no_grad()
    def _update(self, d):
        m = self.momentum
        if not self.initialized:
            self.disp_mean.copy_(d.mean())
            self.disp_std.copy_(d.std() + self.eps)
            self.initialized.fill_(True)
        else:
            self.disp_mean.mul_(1 - m).add_(d.mean(), alpha=m)
            self.disp_std.mul_(1 - m).add_(d.std() + self.eps, alpha=m)

    def forward(self, hidden_states):
        # 마지막 4개 layer의 [CLS] 방향 분산 (초기 layer는 표면적이라 제외)
        cls_layers = torch.stack([h[:, 0] for h in hidden_states[-4:]], dim=1)  # (B, L, H)
        cls_dir = F.normalize(cls_layers, dim=-1)            # 방향만
        mean_dir = F.normalize(cls_dir.mean(dim=1), dim=-1)  # (B, H) 평균 방향

        cos = (cls_dir * mean_dir.unsqueeze(1)).sum(dim=-1).clamp(-1, 1)  # (B, L)
        disp = (1.0 - cos).mean(dim=-1)                      # (B,) 평균 불일치

        if self.training:
            self._update(disp.detach())

        mu = self.disp_mean.detach()
        std = self.disp_std.detach().clamp(min=self.eps)
        ambiguity = torch.sigmoid((disp - mu) / std)
        return ambiguity.detach() if self.detach_out else ambiguity


# ─── 3. EpistemicBERT ─────────────────────────────────────────────────────────
class EpistemicBERT(nn.Module):

    def __init__(
        self,
        n_classes: int = 3,
        n_prototypes: int = 32,
        proj_dim: int = 128,
        freeze_bert: bool = False,
    ):
        super().__init__()

        self.bert = DistilBertModel.from_pretrained("distilbert-base-uncased")
        self.bert.config.output_attentions = True
        self.bert.config.output_hidden_states = True
        if freeze_bert:
            for p in self.bert.parameters():
                p.requires_grad = False

        self.proj = nn.Linear(768, proj_dim)
        self.manifold = PrototypeManifold(proj_dim, n_prototypes)
        self.token_nov = TokenNovelty(self.bert.config.vocab_size)
        self.attn_ign = AttentionIgnorance()
        self.layer_amb = LayerAmbiguity()
        self.epistemic = EpistemicFieldClassifier()

        self.field_proj = nn.Linear(6, n_classes)
        self.z_proj = nn.Linear(proj_dim, n_classes)

        self.margin_param = nn.Parameter(torch.tensor(0.2))
        self.energy_ceiling = nn.Parameter(torch.tensor(0.5))

    def forward(self, input_ids, attention_mask):
        bert_out = self.bert(
            input_ids=input_ids, attention_mask=attention_mask,
            output_attentions=True,
        )
        cls = bert_out.last_hidden_state[:, 0]

        z = self.proj(cls)
        manifold_out = self.manifold(z)
        manifold_out["novelty_score"] = self.token_nov(input_ids, attention_mask)
        manifold_out["ambiguity_score"] = self.layer_amb(bert_out.hidden_states)

        ignorance = self.attn_ign(bert_out.attentions, input_ids, attention_mask)

        epistemic_out = self.epistemic(manifold_out, ignorance)

        field = epistemic_out["field"]
        logits = self.field_proj(field) + self.z_proj(z)

        return logits, epistemic_out, manifold_out["diversity_loss"]


# ─── Config ───────────────────────────────────────────────────────────────────
CFG = dict(
    model=dict(
        n_classes=3,
        n_prototypes=32,
        proj_dim=128,
        freeze_bert=False,
    ),
    train=dict(
        batch_size=32,
        epochs=3,
        lr_bert=2e-5,
        lr_head=1e-3,
        max_length=192,
        lambda_ce=0.3,
        lambda_field=1.0,
        lambda_margin=0.2,
        lambda_diversity=0.1,
        ranking_margin=0.1,
        train_size=30_000,
        val_size=5_000,
        lambda_disent=0.03,
    ),
    device="cuda" if torch.cuda.is_available() else "cpu",
    seed=42,
)


# ─── Dataset ──────────────────────────────────────────────────────────────────
class FEVERDataset(Dataset):
    LABEL_MAP = {"SUPPORTS": 0, "REFUTES": 1, "NOT ENOUGH INFO": 2}

    def __init__(self, hf_split, tokenizer, max_length):
        self.data = hf_split
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        lbl = item["label"]
        lbl = self.LABEL_MAP[lbl] if isinstance(lbl, str) else int(lbl)
        enc = self.tokenizer(
            item["claim"], item["evidence"],          # claim 먼저 → [CLS] claim [SEP] evidence [SEP]
            max_length=self.max_length, padding="max_length",
            truncation=True, return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": torch.tensor(lbl, dtype=torch.long),
            "idx": torch.tensor(idx, dtype=torch.long),   # 원본 정렬 보장 (추출 시 사용; 학습/평가는 무시)
        }


class OODDataset(Dataset):
    def __init__(self, hf_split, tokenizer, max_length):
        self.data = hf_split
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        enc = self.tokenizer(
            item["sentence1"], item["sentence2"],
            max_length=self.max_length, padding="max_length",
            truncation=True, return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
        }


# ─── Loss ─────────────────────────────────────────────────────────────────────
def field_ranking_loss(field, labels, margin: float = 0.1, eps: float = 1e-8):
    loss = torch.tensor(0.0, device=field.device)
    n = torch.tensor(0, device=field.device)

    sup_mask = (labels == 0)   # SUPPORTS → truth(0)
    ref_mask = (labels == 1)   # REFUTES  → error(1)
    nei_mask = (labels == 2)   # NEI      → ignorance(5)

    if sup_mask.any():
        f = field[sup_mask]
        for idx in [1, 2, 3, 4, 5]:
            loss += F.relu(margin - (f[:, 0] - f[:, idx])).mean()
        n += sup_mask.sum()

    if ref_mask.any():
        f = field[ref_mask]
        for idx in [0, 2, 3, 4, 5]:
            loss += F.relu(margin - (f[:, 1] - f[:, idx])).mean()
        n += ref_mask.sum()

    if nei_mask.any():
        f = field[nei_mask]
        for idx in [0, 1, 2, 3, 4]:
            loss += F.relu(margin - (f[:, 5] - f[:, idx])).mean()
        n += nei_mask.sum()

    return loss / (n.float() + eps)


def margin_loss(support, counter, labels, margin_param, energy_ceiling):
    margin = F.softplus(margin_param).clamp(min=0.05)
    ceiling = F.softplus(energy_ceiling).clamp(min=0.2)

    loss = torch.tensor(0.0, device=support.device)
    sup_mask = (labels == 0)   # SUPPORTS: support > counter
    ref_mask = (labels == 1)   # REFUTES : counter > support
    nei_mask = (labels == 2)   # NEI     : 둘 다 낮게 (관련 증거 없음)

    if sup_mask.any():
        loss += F.relu(counter[sup_mask] - support[sup_mask] + margin).mean()

    if ref_mask.any():
        loss += F.relu(support[ref_mask] - counter[ref_mask] + margin).mean()

    if nei_mask.any():
        energy = support[nei_mask] ** 2 + counter[nei_mask] ** 2
        loss += F.relu(energy - ceiling).mean()

    return loss / 3.0


# ─── Optimizer ────────────────────────────────────────────────────────────────
def build_optimizer(model, cfg):
    bert_params = list(model.bert.parameters())
    head_params = (
        list(model.proj.parameters())
        + list(model.manifold.parameters())
        + list(model.epistemic.parameters())
        + list(model.field_proj.parameters())
        + list(model.z_proj.parameters())
        + [model.margin_param, model.energy_ceiling]
    )
    return torch.optim.AdamW([
        {"params": bert_params, "lr": cfg["lr_bert"]},
        {"params": head_params, "lr": cfg["lr_head"]},
    ], weight_decay=1e-2)


# ─── Train ────────────────────────────────────────────────────────────────────
def train_epoch(model, loader, optimizer, scaler, device, tcfg):

    def disentangle_loss(eout):
        def corr(a, b):
            a_c = a - a.mean()
            b_c = b - b.mean()
            return ((a_c * b_c).mean() / (a_c.std() * b_c.std() + 1e-8)).abs()

        nov = eout["novelty"]
        amb = eout["ambiguity"]
        ign = eout["ignorance"]
        return (corr(nov, amb) + corr(amb, ign)) / 2.0

    model.train()
    ce_fn = nn.CrossEntropyLoss()
    total_loss = total_ce = total_field = total_correct = total = 0

    for batch in tqdm(loader, desc="train", mininterval=10.0, ncols=80):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            logits, eout, div_loss = model(input_ids, attention_mask)

            ce = ce_fn(logits, labels)
            f_loss = field_ranking_loss(eout["field"], labels, tcfg["ranking_margin"])
            m_loss = margin_loss(
                eout["support_raw"], eout["counter_raw"], labels,
                model.margin_param, model.energy_ceiling,
            )
            d_loss = disentangle_loss(eout)
            loss = (
                tcfg["lambda_ce"] * ce
                + tcfg["lambda_field"] * f_loss
                + tcfg["lambda_margin"] * m_loss
                + tcfg["lambda_diversity"] * div_loss
                + tcfg["lambda_disent"] * d_loss
            )

        if torch.isnan(loss) or torch.isinf(loss):
            print(f"  nan/inf  ce={ce.item():.4f}  f={f_loss.item():.4f}  m={m_loss.item():.4f}")
            break

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        preds = logits.argmax(dim=-1)
        total_correct += (preds == labels).sum().item()
        total += labels.size(0)
        total_loss += loss.item()
        total_ce += ce.item()
        total_field += f_loss.item()

    n = len(loader)
    return {
        "loss": total_loss / n,
        "ce": total_ce / n,
        "field": total_field / n,
        "acc": total_correct / total,
    }


# ─── Evaluate ─────────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    ce_fn = nn.CrossEntropyLoss()
    AXES = EpistemicFieldClassifier.AXES

    axis_sums = defaultdict(lambda: defaultdict(float))
    axis_counts = defaultdict(int)
    s_sums = defaultdict(float)
    c_sums = defaultdict(float)
    total_loss = total_correct = total = 0
    field_all = []

    for batch in tqdm(loader, desc="eval", mininterval=10.0, ncols=80):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        with torch.cuda.amp.autocast():
            logits, eout, _ = model(input_ids, attention_mask)
            loss = ce_fn(logits, labels)

        preds = logits.argmax(dim=-1)
        total_correct += (preds == labels).sum().item()
        total_loss += loss.item()
        total += labels.size(0)
        field_all.append(eout["field"].cpu().float())

        for b in range(labels.size(0)):
            lbl = labels[b].item()
            axis_counts[lbl] += 1
            s_sums[lbl] += eout["support_raw"][b].item()
            c_sums[lbl] += eout["counter_raw"][b].item()
            for a, ax in enumerate(AXES):
                axis_sums[lbl][ax] += eout["field"][b, a].item()

    label_names = {0: "SUPPORTS", 1: "REFUTES", 2: "NEI"}
    field_by_label = {
        label_names[lbl]: {ax: axis_sums[lbl][ax] / axis_counts[lbl] for ax in AXES}
        for lbl in label_names if axis_counts[lbl] > 0
    }
    field_cat = torch.cat(field_all, dim=0)
    monitor = {
        "field_mean": field_cat.mean(0).numpy().round(4).tolist(),
        "field_std": field_cat.std(0).numpy().round(4).tolist(),
    }
    return {
        "loss": total_loss / len(loader),
        "acc": total_correct / total,
        "field_by_label": field_by_label,
        "monitor": monitor,
        "support_by_label": {
            label_names[lbl]: {
                "support": s_sums[lbl] / axis_counts[lbl],
                "counter": c_sums[lbl] / axis_counts[lbl],
            }
            for lbl in label_names if axis_counts[lbl] > 0
        },
    }


@torch.no_grad()
def evaluate_ood(model, ood_loader, id_loader, device):
    model.eval()
    AXES = EpistemicFieldClassifier.AXES

    def collect(loader):
        fs = []
        for batch in tqdm(loader, desc="ood", mininterval=10.0, ncols=80):
            with torch.cuda.amp.autocast():
                _, eout, _ = model(
                    batch["input_ids"].to(device),
                    batch["attention_mask"].to(device),
                )
            fs.append(eout["field"].cpu())
        return torch.cat(fs, dim=0)

    id_f = collect(id_loader)
    ood_f = collect(ood_loader)
    return {
        ax: {"id_mean": id_f[:, i].mean().item(), "ood_mean": ood_f[:, i].mean().item()}
        for i, ax in enumerate(AXES)
    }


@torch.no_grad()
def identifiability_probe(model, loader, device):
    model.eval()
    AXES = EpistemicFieldClassifier.AXES

    fields = []
    for batch in tqdm(loader, desc="ident", mininterval=10.0, ncols=80):
        _, eout, _ = model(
            batch["input_ids"].to(device),
            batch["attention_mask"].to(device),
        )
        fields.append(eout["field"].cpu().float())
    F_mat = torch.cat(fields, dim=0).numpy()

    # plane 축(0,1,2)은 redundant 정상, uncertainty 축(3,4,5)만 independent 기대
    print(f"\n{'axis':>16}  {'R² from others':>16}  {'group':>10}  {'verdict':>14}")
    print("─" * 62)
    groups = {0: "plane", 1: "plane", 2: "plane", 3: "uncert", 4: "uncert", 5: "uncert"}
    for i, ax in enumerate(AXES):
        y = F_mat[:, i]
        X = np.delete(F_mat, i, axis=1)
        r2 = LinearRegression().fit(X, y).score(X, y)
        if groups[i] == "plane":
            verdict = "plane-coord (OK)" if r2 > 0.6 else "unexpected-indep"
        else:
            verdict = "LEAK" if r2 > 0.6 else "independent (OK)"
        print(f"{ax:>16}  {r2:>16.4f}  {groups[i]:>10}  {verdict:>14}")

    corr = np.corrcoef(F_mat.T)
    print(f"\n  |correlation| matrix:")
    print(f"{'':>14}" + "".join(f"{a[:5]:>8}" for a in AXES))
    for i, ax in enumerate(AXES):
        row = "".join(f"{abs(corr[i, j]):>8.3f}" for j in range(6))
        print(f"{ax:>14}{row}")


@torch.no_grad()
def diagnose_ambiguity(model, loader, device):
    """ambiguity가 plane으로 새는 게 정규화 포화 때문인지, 소스 본질 결함인지 진단."""
    model.eval()
    la = model.layer_amb

    print(f"\n  LayerAmbiguity EMA 버퍼:")
    print(f"    disp_mean = {la.disp_mean.item():.4f}")
    print(f"    disp_std  = {la.disp_std.item():.4f}   (< 0.03이면 포화 의심)")
    print(f"    initialized = {la.initialized.item()}")

    raw_disps, amb_out, errs, contras = [], [], [], []
    for batch in loader:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        with torch.cuda.amp.autocast():
            out = model.bert(input_ids=ids, attention_mask=mask)
            _, eout, _ = model(ids, mask)
        hs = out.hidden_states
        cls_layers = torch.stack([h[:, 0] for h in hs[-4:]], dim=1)
        cls_dir = F.normalize(cls_layers.float(), dim=-1)
        mean_dir = F.normalize(cls_dir.mean(dim=1), dim=-1)
        cos = (cls_dir * mean_dir.unsqueeze(1)).sum(dim=-1).clamp(-1, 1)
        disp = (1.0 - cos).mean(dim=-1)
        raw_disps.append(disp.cpu().numpy())
        amb_out.append(eout["ambiguity"].float().cpu().numpy())
        errs.append(eout["error"].float().cpu().numpy())
        contras.append(eout["contradiction"].float().cpu().numpy())

    raw = np.concatenate(raw_disps)
    amb = np.concatenate(amb_out)
    err = np.concatenate(errs)
    con = np.concatenate(contras)

    print(f"\n  raw disp (정규화 전, task-free 신호):")
    print(f"    mean={raw.mean():.4f}  std={raw.std():.4f}  min={raw.min():.4f}  max={raw.max():.4f}")
    print(f"  ambiguity (정규화 후, field):")
    print(f"    mean={amb.mean():.4f}  std={amb.std():.4f}   (< 0.05면 거의 상수=포화)")

    def corr(a, b):
        return abs(np.corrcoef(a, b)[0, 1])
    print(f"\n  ── raw disp의 plane 상관 (소스 본질 결함 여부) ──")
    print(f"    raw_disp ↔ error  = {corr(raw, err):.4f}")
    print(f"    raw_disp ↔ contra = {corr(raw, con):.4f}")
    print(f"  ── 정규화 후 ambiguity의 plane 상관 ──")
    print(f"    ambiguity ↔ error  = {corr(amb, err):.4f}")
    print(f"    ambiguity ↔ contra = {corr(amb, con):.4f}")
    print(f"\n  판정: raw도 높으면(>0.5) 소스 본질 결함 / raw 낮고 정규화 후만 높으면 EMA 포화")


@torch.no_grad()
def evaluate_selective_prediction(model, loader, device):
    """by-design 확인용: uncertainty 축은 confidence-independent라 risk-coverage에서 baseline에 짐."""
    model.eval()

    correct_all = []
    acc = {
        "entropy (baseline)": [],
        "1-MSP (baseline)": [],
        "ambiguity (ours)": [],
        "error (ours)": [],
        "amb+error (ours)": [],
        "amb+ign+nov (ours)": [],
        "all-unc+error (ours)": [],
    }

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        with torch.cuda.amp.autocast():
            logits, eout, _ = model(input_ids, attention_mask)

        prob = F.softmax(logits.float(), dim=-1)
        preds = logits.argmax(dim=-1)
        n_cls = prob.size(-1)

        amb = eout["ambiguity"].float()
        ign = eout["ignorance"].float()
        nov = eout["novelty"].float()
        err = eout["error"].float()

        ent = -(prob * (prob + 1e-8).log()).sum(dim=-1)
        ent = ent / torch.log(torch.tensor(float(n_cls), device=device))
        msp = 1.0 - prob.max(dim=-1).values

        correct_all.append((preds == labels).cpu())
        acc["entropy (baseline)"].append(ent.cpu())
        acc["1-MSP (baseline)"].append(msp.cpu())
        acc["ambiguity (ours)"].append(amb.cpu())
        acc["error (ours)"].append(err.cpu())
        acc["amb+error (ours)"].append((amb + err).cpu())
        acc["amb+ign+nov (ours)"].append((amb + ign + nov).cpu())
        acc["all-unc+error (ours)"].append((amb + ign + nov + err).cpu())

    correct = torch.cat(correct_all).numpy().astype(float)
    scores = {name: torch.cat(parts).numpy() for name, parts in acc.items()}
    return _risk_coverage_report(correct, scores)


def _risk_coverage_report(correct, scores):
    N = len(correct)
    base_risk = 1.0 - correct.mean()

    print(f"\n  base error rate (coverage=100%): {base_risk:.4f}")
    print(f"\n{'method':>24}  {'AURC':>8}  {'risk@90%':>9}  {'risk@80%':>9}  {'risk@70%':>9}")
    print("─" * 68)

    results = {}
    for name, unc in scores.items():
        order = np.argsort(unc)            # 확신 높은(불확실 낮은) 것부터
        c_sorted = correct[order]
        risks = 1.0 - np.cumsum(c_sorted) / np.arange(1, N + 1)
        aurc = risks.mean()

        def risk_at(cov):
            k = max(1, int(cov * N))
            return 1.0 - c_sorted[:k].mean()

        results[name] = {"aurc": aurc, "risk90": risk_at(0.9),
                         "risk80": risk_at(0.8), "risk70": risk_at(0.7)}
        print(f"{name:>24}  {aurc:>8.4f}  {results[name]['risk90']:>9.4f}"
              f"  {results[name]['risk80']:>9.4f}  {results[name]['risk70']:>9.4f}")

    base_keys = [k for k in scores if "baseline" in k]
    ours_keys = [k for k in scores if "ours" in k]
    best_base = min(results[k]["aurc"] for k in base_keys)
    best_ours_key = min(ours_keys, key=lambda k: results[k]["aurc"])
    best_ours = results[best_ours_key]["aurc"]

    print(f"\n  best baseline AURC = {best_base:.4f}")
    print(f"  best ours AURC     = {best_ours:.4f}  ({best_ours_key})")
    print(f"  → {'OURS WINS' if best_ours < best_base else 'baseline wins'} "
          f"(Δ={best_base - best_ours:+.4f})")
    return results


@torch.no_grad()
def failure_typing(model, loader, tokenizer, device, conf_thresh=0.95, k_examples=3):
    """메인 결과: confident wrong을 6축으로 유형 분리.
    silent-wrong + high-conf Δ(uncertainty 축이 안 움직임)도 함께 보고."""
    model.eval()
    AXES = EpistemicFieldClassifier.AXES
    unc_idx = {"novelty": 3, "ambiguity": 4, "ignorance": 5}

    rows = []   # (conf, ent, field[6], correct, ids, label, pred)
    for batch in loader:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        lbl = batch["label"].to(device)
        with torch.cuda.amp.autocast():
            logits, eout, _ = model(ids, mask)
        prob = F.softmax(logits.float(), dim=-1)
        preds = logits.argmax(dim=-1)
        conf = prob.max(dim=-1).values
        ent = -(prob * (prob + 1e-8).log()).sum(-1) / torch.log(
            torch.tensor(float(prob.size(-1)), device=device))
        fld = eout["field"].float()
        for b in range(lbl.size(0)):
            rows.append((
                conf[b].item(), ent[b].item(), fld[b].cpu().numpy(),
                (preds[b] == lbl[b]).item(),
                ids[b].cpu(), lbl[b].item(), preds[b].item(),
            ))

    conf_a = np.array([r[0] for r in rows])
    ent_a = np.array([r[1] for r in rows])
    fld_a = np.stack([r[2] for r in rows])
    corr_a = np.array([r[3] for r in rows], dtype=bool)
    uncert = list(unc_idx.values())

    # ── silent wrong: 틀렸는데 uncertainty 축이 안 뜬 비율 ──
    thresh = np.median(fld_a[corr_a][:, uncert].max(axis=1))
    wrong_umax = fld_a[~corr_a][:, uncert].max(axis=1)
    silent = wrong_umax < thresh
    print(f"\n  silent wrong (틀렸으나 uncertainty 잠잠) = {silent.mean():.4f} "
          f"({silent.sum()}/{len(silent)})  [thresh={thresh:.4f}]")

    # ── high-conf wrong: uncertainty 축이 안 움직인다는 측정 사실 ──
    hc_c = corr_a & (conf_a > conf_thresh)
    hc_w = (~corr_a) & (conf_a > conf_thresh)
    if hc_c.sum() > 0 and hc_w.sum() > 0:
        dc = fld_a[hc_w].mean(0) - fld_a[hc_c].mean(0)
        print(f"  high-conf(>{conf_thresh}) wrong vs correct Δ:  "
              f"novelty={dc[3]:+.4f}  ambiguity={dc[4]:+.4f}  ignorance={dc[5]:+.4f}")

    # ── confident wrong 유형 분리 ──
    cw = hc_w
    n_cw = int(cw.sum())
    print(f"\n  confident wrong (conf>{conf_thresh}): {n_cw}개")
    if n_cw < 6:
        print("  표본 너무 적음 — conf_thresh 낮추거나 epoch 더.")
        return {"silent": float(silent.mean())}

    cw_fld, cw_conf, cw_ent = fld_a[cw], conf_a[cw], ent_a[cw]
    LOW = np.median(fld_a[corr_a][:, uncert], axis=0)
    types = []
    for s in cw_fld:
        u = {ax: s[i] for ax, i in unc_idx.items()}
        top_ax = max(u, key=u.get)
        types.append(top_ax if u[top_ax] > LOW[list(unc_idx).index(top_ax)] else "metacognitive")
    types = np.array(types)

    print(f"\n{'유형':>14}  {'n':>4}  {'conf':>7}  {'entropy':>8}  "
          f"{'novel':>7}  {'ambig':>7}  {'ignor':>7}  {'error':>7}")
    print("─" * 76)
    for t in ["ignorance", "ambiguity", "novelty", "metacognitive"]:
        m = types == t
        if m.sum() == 0:
            continue
        f = cw_fld[m]
        print(f"{t:>14}  {m.sum():>4}  {cw_conf[m].mean():>7.3f}  {cw_ent[m].mean():>8.3f}  "
              f"{f[:,3].mean():>7.3f}  {f[:,4].mean():>7.3f}  {f[:,5].mean():>7.3f}  {f[:,1].mean():>7.3f}")

    print(f"\n  ── conf/entropy로 유형 분리되나 (안 돼야 우리 주장 성립) ──")
    for t in ["ignorance", "ambiguity", "metacognitive"]:
        m = types == t
        if m.sum() > 0:
            print(f"    {t:>14}: conf={cw_conf[m].mean():.3f}±{cw_conf[m].std():.3f}  "
                  f"ent={cw_ent[m].mean():.3f}±{cw_ent[m].std():.3f}")

    LABELS = {0: "SUPPORTS", 1: "REFUTES", 2: "NEI"}
    print(f"\n  ── 유형별 실제 사례 (claim ‖ evidence) ──")
    cw_rows = [r for r, m in zip(rows, cw) if m]
    for t in ["ignorance", "ambiguity", "metacognitive"]:
        idxs = np.where(types == t)[0]
        if len(idxs) == 0:
            continue
        order = (idxs[np.argsort(cw_conf[idxs])[::-1]] if t == "metacognitive"
                 else idxs[np.argsort(cw_fld[idxs][:, unc_idx[t]])[::-1]])
        print(f"\n  [{t}]")
        for j in order[:k_examples]:
            r = cw_rows[j]
            text = tokenizer.decode(r[4], skip_special_tokens=True)
            text = text[:160] + ("…" if len(text) > 160 else "")
            print(f"    gold={LABELS[r[5]]:>8} pred={LABELS[r[6]]:>8} conf={r[0]:.3f}  "
                  f"nov={r[2][3]:.2f} amb={r[2][4]:.2f} ign={r[2][5]:.2f}")
            print(f"      {text}")

    return {"types": types, "cw_fld": cw_fld, "cw_conf": cw_conf, "silent": float(silent.mean())}


@torch.no_grad()
def evaluate_fever_axes(model, loader, device):
    model.eval()
    AXES = EpistemicFieldClassifier.AXES
    LABELS = {0: "SUPPORTS", 1: "REFUTES", 2: "NEI"}
    fields, labels = [], []
    for batch in loader:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        with torch.cuda.amp.autocast():
            _, eout, _ = model(ids, mask)
        fields.append(eout["field"].float().cpu())
        labels.append(batch["label"])
    fld = torch.cat(fields).numpy()
    lab = torch.cat(labels).numpy()
    means = {l: fld[lab == l].mean(0) for l in [0, 1, 2] if (lab == l).any()}

    print(f"\n{'axis':>14}" + "".join(f"{LABELS[l]:>11}" for l in [0, 1, 2]))
    print("─" * 50)
    for i, ax in enumerate(AXES):
        row = "".join(f"{means[l][i]:>11.4f}" if l in means else f"{'-':>11}" for l in [0, 1, 2])
        print(f"{ax:>14}{row}")

    def hi(axis_i, target):
        vals = {l: means[l][axis_i] for l in means}
        top = max(vals, key=vals.get)
        return f"{'✓' if top == target else '✗'} (top={LABELS[top]} {vals[top]:.3f})"

    print(f"\n  매핑 점검:")
    print(f"    truth     → SUPPORTS  {hi(0, 0)}")
    print(f"    error     → REFUTES   {hi(1, 1)}")
    print(f"    ignorance → NEI       {hi(5, 2)}")

    print(f"\n  |corr| ignorance vs 나머지 (분리 확인):")
    for i, ax in enumerate(AXES):
        if ax == "ignorance":
            continue
        c = abs(np.corrcoef(fld[:, 5], fld[:, i])[0, 1])
        print(f"    ignorance ↔ {ax:>14}: {c:.4f}")
    return means


# ─── Pretty Print ─────────────────────────────────────────────────────────────
def print_field_by_label(d):
    AXES = EpistemicFieldClassifier.AXES
    print(f"{'':>15}" + "".join(f"{a:>14}" for a in AXES))
    print("─" * (15 + 14 * len(AXES)))
    for name, vals in d.items():
        print(f"{name:>15}" + "".join(f"{vals[a]:>14.4f}" for a in AXES))


def print_ood_comparison(r):
    AXES = EpistemicFieldClassifier.AXES
    print(f"\n{'axis':>16}  {'ID mean':>10}  {'OOD mean':>10}  {'OOD - ID':>10}")
    print("─" * 52)
    for ax in AXES:
        id_m, ood_m = r[ax]["id_mean"], r[ax]["ood_mean"]
        flag = "  ← ↑" if ax in ("novelty", "ignorance", "ambiguity") and (ood_m - id_m) > 0.05 else ""
        print(f"{ax:>16}  {id_m:>10.4f}  {ood_m:>10.4f}  {ood_m - id_m:>+10.4f}{flag}")


# ─── Main ─────────────────────────────────────────────────────────────────────
def main(seed=None, ablation="none"):
    """ablation: "none" | "dispersion"(cross-attn→entropy) | "no_detach"(소스 detach 제거)."""
    if seed is None:
        seed = CFG["seed"]
    import random
    torch.manual_seed(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed_all(seed)
    device = CFG["device"]
    tcfg = CFG["train"]
    print(f"Device: {device}  |  seed={seed}")

    tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

    print("Loading VitaminC...")
    vc = load_dataset("tals/vitaminc")
    train_split = vc["train"].select(range(min(tcfg["train_size"], len(vc["train"])))) if tcfg["train_size"] else vc["train"]
    val_split = vc["validation"].select(range(min(tcfg["val_size"], len(vc["validation"])))) if tcfg["val_size"] else vc["validation"]

    train_ds = FEVERDataset(train_split, tokenizer, tcfg["max_length"])
    val_ds = FEVERDataset(val_split, tokenizer, tcfg["max_length"])
    train_loader = DataLoader(train_ds, batch_size=tcfg["batch_size"], shuffle=True, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=tcfg["batch_size"], shuffle=False, num_workers=0, pin_memory=True)

    print("Loading RTE (OOD)...")
    rte = load_dataset("glue", "rte")
    ood_ds = OODDataset(rte["validation"], tokenizer, tcfg["max_length"])
    ood_loader = DataLoader(ood_ds, batch_size=tcfg["batch_size"], shuffle=False, num_workers=0, pin_memory=True)

    model = EpistemicBERT(**CFG["model"]).to(device)
    model.token_nov.set_special_tokens(tokenizer.all_special_ids)
    model.attn_ign.sep_id = tokenizer.sep_token_id

    # ── ablation 스위치 ──────────────────────────────────────────
    if ablation == "dispersion":
        model.attn_ign.mode = "dispersion"
    elif ablation == "no_detach":
        model.token_nov.detach_out = False
        model.layer_amb.detach_out = False
        model.attn_ign.detach_out = False
    print(f"  ablation = {ablation}")
    print(f"Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    optimizer = build_optimizer(model, tcfg)
    scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda"))

    history = []
    for epoch in range(tcfg["epochs"]):
        print(f"\n═══ Epoch {epoch + 1}/{tcfg['epochs']} ═══")

        tr = train_epoch(model, train_loader, optimizer, scaler, device, tcfg)
        print(f"  train  loss={tr['loss']:.4f}  ce={tr['ce']:.4f}  field={tr['field']:.4f}  acc={tr['acc']:.4f}")
        print(f"  learnable params  margin={F.softplus(model.margin_param).item():.4f}  "
              f"energy_ceil={F.softplus(model.energy_ceiling).item():.4f}  "
              f"truth_temp={F.softplus(model.epistemic.truth_temp).item():.4f}")

        vl = evaluate(model, val_loader, device)
        m = vl["monitor"]
        print(f"  monitor  field_mean={m['field_mean']}")
        print(f"           field_std ={m['field_std']}")
        print(f"  val    loss={vl['loss']:.4f}  acc={vl['acc']:.4f}")

        print("\n  Epistemic field by label (val):")
        print_field_by_label(vl["field_by_label"])
        print("\n  Support / Counter by label:")
        for name, vals in vl["support_by_label"].items():
            print(f"  {name:>15}  support={vals['support']:.4f}  counter={vals['counter']:.4f}")

        history.append({"epoch": epoch + 1, "train": tr, "val": vl})

    print("\n\n═══ OOD Experiment ═══")
    print_ood_comparison(evaluate_ood(model, ood_loader, val_loader, device))

    print("\n\n═══ Identifiability Probe ═══")
    identifiability_probe(model, val_loader, device)

    print("\n\n═══ Ambiguity 진단 ═══")
    diagnose_ambiguity(model, val_loader, device)

    print("\n\n═══ Selective Prediction (by-design 확인) ═══")
    evaluate_selective_prediction(model, val_loader, device)

    print("\n\n═══ Failure Typing (메인 결과) ═══")
    ft = failure_typing(model, val_loader, tokenizer, device)

    print("\n\n═══ FEVER Axis ↔ Label Mapping ═══")
    fever_means = evaluate_fever_axes(model, val_loader, device)

    save_path = f"/kaggle/working/epistemic_bert_seed{seed}.pt"
    torch.save(model.state_dict(), save_path)
    print(f"  saved \u2192 {save_path}")
    with open(f"/kaggle/working/results_seed{seed}.json", "w") as f:
        json.dump({"history": history}, f, indent=2)

    return model, val_loader, device, ft, fever_means


RUN_TRAINING = False   # ← True로 바꿀 때만 학습 실행

if __name__ == "__main__" and RUN_TRAINING:
    for s in [42, 7, 123]:
        print("\n" + "█" * 70)
        print(f"█  SEED {s}")
        print("█" * 70)
        main(seed=s, ablation="none")

In [5]:
# ── 이전 셀에서 epistemic_bert.py 를 먼저 실행해야 함 ──────────────────────
# (CFG, EpistemicBERT, FEVERDataset, build_optimizer 등이 이미 정의됨)

"""
Phase 1 — Baseline 비교 (evidential DL + MC dropout).

목적: "confident error를 유형(I/A/M)으로 가르는 게 우리 6축 고유 능력인가,
       아니면 기존 uncertainty 방법도 하는가?"

공정성:
  - 같은 backbone (DistilBERT-base), 같은 데이터 (VitaminC), 같은 epoch/seed
  - evidential: 처음부터 재학습 (Sensoy 2018 표준: Dirichlet + Bayes risk + KL anneal)
  - MC dropout: 우리 모델에 dropout 켜고 추론 ×N (재학습 없음)

핵심 테스트:
  evidential의 uncertainty u 가 우리 사람-라벨 I형(증거부족)과 M형(판정실패)을
  가르나, 뭉개나? confident error라 둘 다 u 낮으면 → 못 가름 = 우리가 채우는 갭.

출력:
  baseline_scores.csv — confident error id별 evidential u, MC dropout variance.
  (cw_answer_key.csv 의 id 와 정렬 → 나중에 사람 라벨과 합쳐 분석)
"""

import csv
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import DistilBertModel, DistilBertTokenizerFast
from datasets import load_dataset


# ══════════════════════════════════════════════════════════════════════════════
# Part A — Evidential Deep Learning (Sensoy et al. 2018)
# ══════════════════════════════════════════════════════════════════════════════

class EvidentialBERT(nn.Module):
    """DistilBERT + evidential head. 우리 모델과 같은 backbone, head만 다름.

    logits 대신 evidence(≥0) 출력 → Dirichlet α = evidence + 1.
    uncertainty u = K / S,  S = Σα.
    """
    def __init__(self, n_classes=3):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained("distilbert-base-uncased")
        self.n_classes = n_classes
        self.head = nn.Linear(768, n_classes)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0]
        # evidence ≥ 0 (softplus). exp 대신 softplus가 안정적 (Sensoy 권장)
        evidence = F.softplus(self.head(cls))     # (B, K)
        alpha = evidence + 1.0                     # Dirichlet param
        return evidence, alpha

    @staticmethod
    def uncertainty(alpha):
        """u = K / S. evidence 적을수록 1에 가까움 (= epistemic uncertainty)."""
        K = alpha.size(-1)
        S = alpha.sum(dim=-1)
        return K / S                               # (B,)

    @staticmethod
    def prob(alpha):
        """expected class probability = α / S."""
        return alpha / alpha.sum(dim=-1, keepdim=True)


def evidential_loss(alpha, target, epoch, total_epochs, n_classes=3):
    """Bayes risk (MSE form) + KL 정규화 (annealed).

    Sensoy 2018 Eq.5 (MSE) + Eq.9 (KL). KL coef는 epoch에 따라 0→1 anneal
    (초반에 KL 너무 세면 학습 안 됨).
    """
    S = alpha.sum(dim=-1, keepdim=True)
    p = alpha / S
    y = F.one_hot(target, n_classes).float()

    # Bayes risk (expected sum of squares): Σ (y - p)² + p(1-p)/(S+1)
    err = ((y - p) ** 2).sum(dim=-1)
    var = (p * (1 - p) / (S + 1)).sum(dim=-1)
    mse = err + var

    # KL(Dirichlet(α~) || Dirichlet(1)) — 틀린 클래스 evidence를 0으로
    alpha_tilde = y + (1 - y) * alpha              # 정답 클래스는 1로 고정
    kl = _kl_dirichlet_uniform(alpha_tilde, n_classes)

    anneal = min(1.0, epoch / max(1, total_epochs // 2))   # 절반까지 0→1
    return (mse + anneal * 0.1 * kl).mean()


def _kl_dirichlet_uniform(alpha, K):
    """KL( Dir(alpha) || Dir(1,...,1) ).  alpha=1(균등)이면 0."""
    S = alpha.sum(dim=-1, keepdim=True)
    # ln B(alpha) = Sum lnGamma(a_k) - lnGamma(S)
    ln_b_alpha = torch.lgamma(alpha).sum(dim=-1, keepdim=True) - torch.lgamma(S)
    # ln B(1..1) = Sum lnGamma(1) - lnGamma(K) = -lnGamma(K)
    ln_b_uniform = -torch.lgamma(torch.tensor(float(K), device=alpha.device))
    # Sum (a_k - 1)(psi(a_k) - psi(S))
    term = ((alpha - 1) * (torch.digamma(alpha) - torch.digamma(S))).sum(dim=-1, keepdim=True)
    # KL = ln B(uniform) - ln B(alpha) + term
    kl = ln_b_uniform - ln_b_alpha + term
    return kl.squeeze(-1)


def train_evidential(seed=42, ckpt_out="/kaggle/working/evidential_seed42.pt"):
    """VitaminC로 evidential 모델 재학습 (우리 모델과 같은 조건)."""
    import random
    torch.manual_seed(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed_all(seed)
    device = CFG["device"]
    tcfg = CFG["train"]

    tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
    vc = load_dataset("tals/vitaminc")
    train_split = vc["train"].select(range(min(tcfg["train_size"], len(vc["train"]))))
    val_split = vc["validation"].select(range(min(tcfg["val_size"], len(vc["validation"]))))

    train_ds = FEVERDataset(train_split, tokenizer, tcfg["max_length"])
    val_ds = FEVERDataset(val_split, tokenizer, tcfg["max_length"])
    train_loader = DataLoader(train_ds, batch_size=tcfg["batch_size"], shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=tcfg["batch_size"], shuffle=False)

    model = EvidentialBERT(n_classes=3).to(device)
    # 우리 모델과 동일한 lr 구조 (backbone 2e-5, head 1e-3)
    optimizer = torch.optim.AdamW([
        {"params": model.bert.parameters(), "lr": tcfg["lr_bert"]},
        {"params": model.head.parameters(), "lr": tcfg["lr_head"]},
    ], weight_decay=1e-2)
    scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda"))

    for epoch in range(tcfg["epochs"]):
        model.train()
        tot_loss = tot_correct = tot = 0
        for batch in train_loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            lbl = batch["label"].to(device)
            optimizer.zero_grad()
            with torch.cuda.amp.autocast():
                evidence, alpha = model(ids, mask)
                loss = evidential_loss(alpha, lbl, epoch, tcfg["epochs"])
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            pred = alpha.argmax(-1)
            tot_correct += (pred == lbl).sum().item()
            tot += lbl.size(0)
            tot_loss += loss.item()
        print(f"  evidential epoch {epoch+1}: loss={tot_loss/len(train_loader):.4f} "
              f"acc={tot_correct/tot:.4f}")

    # val acc (공정성 확인 — 우리 모델 acc와 비슷해야 비교 정당)
    model.eval()
    vc_correct = vt = 0
    with torch.no_grad():
        for batch in val_loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            lbl = batch["label"].to(device)
            with torch.cuda.amp.autocast():
                _, alpha = model(ids, mask)
            vc_correct += (alpha.argmax(-1) == lbl).sum().item()
            vt += lbl.size(0)
    print(f"  evidential val acc = {vc_correct/vt:.4f}  (우리 모델 ~0.63과 비교)")

    torch.save(model.state_dict(), ckpt_out)
    print(f"  saved → {ckpt_out}")
    return model


# ══════════════════════════════════════════════════════════════════════════════
# Part B — MC Dropout (우리 모델에 dropout 켜고 추론 ×N)
# ══════════════════════════════════════════════════════════════════════════════

def enable_dropout(model):
    """eval 모드에서도 dropout layer만 train 모드로 (MC dropout 핵심)."""
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.train()


@torch.no_grad()
def mc_dropout_uncertainty(model, ids, mask, n_samples=20):
    """dropout 켜고 N번 추론 → predictive variance + mean entropy.

    반환: (variance, mean_entropy) 둘 다 (B,)
      variance: N개 softmax의 분산 합 (epistemic 근사)
      mean_entropy: 평균 softmax의 entropy (total uncertainty)
    """
    model.eval()
    enable_dropout(model)          # dropout만 다시 켬

    probs = []
    for _ in range(n_samples):
        with torch.cuda.amp.autocast():
            logits, _, _ = model(ids, mask)
        probs.append(F.softmax(logits.float(), dim=-1))
    probs = torch.stack(probs, dim=0)          # (N, B, K)

    mean_p = probs.mean(dim=0)                  # (B, K)
    # predictive variance: 클래스별 분산의 합
    var = probs.var(dim=0).sum(dim=-1)          # (B,)
    # mean entropy
    ent = -(mean_p * (mean_p + 1e-8).log()).sum(dim=-1)
    return var, ent


# ══════════════════════════════════════════════════════════════════════════════
# Part C — confident error id 별 baseline 점수 추출
# ══════════════════════════════════════════════════════════════════════════════

@torch.no_grad()
def extract_baseline_scores(
        our_ckpt_fmt="/kaggle/input/datasets/terryterry9/epistemicbert-3seed/epistemic_bert_seed{}.pt",
        evidential_ckpt="/kaggle/working/evidential_seed42.pt",
        seeds=(42, 7, 123),
        conf_thresh=0.95,
        mc_samples=20,
        out_csv="/kaggle/working/baseline_scores.csv"):
    """cw_answer_key.csv 의 confident error 들에 대해 baseline 점수 계산.

    각 confident error(우리 모델 기준)에 대해:
      - evidential u (해당 입력을 evidential 모델에 통과)
      - MC dropout variance / entropy (우리 모델에 dropout 추론)
    → baseline_scores.csv (id, seed, evidential_u, mc_var, mc_ent)

    id 는 cw_for_labeling.csv / cw_answer_key.csv 와 동일 순서로 정렬됨.
    사람 라벨 채워지면 셋을 id 로 join 해서 분석.
    """
    device = CFG["device"]
    tcfg = CFG["train"]
    tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

    vc = load_dataset("tals/vitaminc")
    val_split = vc["validation"].select(range(min(tcfg["val_size"], len(vc["validation"]))))
    val_ds = FEVERDataset(val_split, tokenizer, tcfg["max_length"])
    val_loader = DataLoader(val_ds, batch_size=tcfg["batch_size"], shuffle=False)

    # evidential 모델 로드
    ev_model = EvidentialBERT(n_classes=3).to(device)
    ev_model.load_state_dict(torch.load(evidential_ckpt, map_location=device))
    ev_model.eval()

    rows = []   # (id, seed, raw_idx, our_conf, ev_u, mc_var, mc_ent)
    global_id = 0
    for s in seeds:
        # 우리 모델 (MC dropout + confident error 판정용)
        our_model = EpistemicBERT(**CFG["model"]).to(device)
        our_model.token_nov.set_special_tokens(tokenizer.all_special_ids)
        our_model.attn_ign.sep_id = tokenizer.sep_token_id
        our_model.load_state_dict(torch.load(our_ckpt_fmt.format(s), map_location=device))
        our_model.eval()

        for batch in val_loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            lbl = batch["label"].to(device)
            bidx = batch["idx"].to(device)

            # 우리 모델 confident error 판정
            with torch.cuda.amp.autocast():
                logits, _, _ = our_model(ids, mask)
            prob = F.softmax(logits.float(), dim=-1)
            pred = logits.argmax(-1)
            conf = prob.max(-1).values

            cw_mask = (pred != lbl) & (conf > conf_thresh)
            if not cw_mask.any():
                continue

            cw_ids = ids[cw_mask]
            cw_masks = mask[cw_mask]
            cw_bidx = bidx[cw_mask]
            cw_conf = conf[cw_mask]

            # evidential u (같은 입력)
            with torch.cuda.amp.autocast():
                _, alpha = ev_model(cw_ids, cw_masks)
            ev_u = EvidentialBERT.uncertainty(alpha.float())

            # MC dropout (우리 모델)
            mc_var, mc_ent = mc_dropout_uncertainty(our_model, cw_ids, cw_masks, mc_samples)
            our_model.eval()   # 복구

            for j in range(cw_ids.size(0)):
                rows.append((global_id, s, int(cw_bidx[j].item()),
                             cw_conf[j].item(), ev_u[j].item(),
                             mc_var[j].item(), mc_ent[j].item()))
                global_id += 1
        print(f"  seed {s}: baseline 점수 추출 완료 (누적 {global_id}개)")

    with open(out_csv, "w", newline="") as fo:
        w = csv.writer(fo)
        w.writerow(["id", "seed", "raw_idx", "our_conf",
                    "evidential_u", "mc_var", "mc_ent"])
        w.writerows(rows)
    print(f"\nbaseline 점수 → {out_csv}  ({len(rows)}개)")

    # 빠른 진단: evidential u 가 confident error 에서 어떤 분포인지
    ev_us = np.array([r[4] for r in rows])
    print(f"\n  evidential u (confident error): "
          f"mean={ev_us.mean():.4f}  std={ev_us.std():.4f}  "
          f"min={ev_us.min():.4f}  max={ev_us.max():.4f}")
    print(f"  → u 가 낮고 분산 작으면: evidential도 confident error를 '확신'으로 봄")
    print(f"     (= 유형 구분 불가 가능성. 사람 라벨과 합쳐서 최종 확인)")
    return out_csv


# ══════════════════════════════════════════════════════════════════════════════
# Part D — 분석 (사람 라벨 채워진 뒤 실행)
# ══════════════════════════════════════════════════════════════════════════════

def analyze_vs_human(
        human_csv="/kaggle/working/cw_for_labeling.csv",   # human_type 채워진 것
        key_csv="/kaggle/input/datasets/terryterry9/epistemicbert-3seed/cw_answer_key.csv",
        baseline_csv="/kaggle/working/baseline_scores.csv"):
    """사람 라벨(I/A/M) ground truth 대비 세 방법의 복원율 비교.

    ⚠️ 사람 라벨링(human_type 채우기)이 끝난 뒤에 실행.
    각 방법:
      - ours: auto_type (cw_answer_key.csv)  vs human
      - evidential: u 로 I/A/M 분리되나
      - MC dropout: var 로 I/A/M 분리되나
    metric: Accuracy / Macro-F1 / Cohen's κ + confusion matrix.
    """
    import pandas as pd
    from sklearn.metrics import (accuracy_score, f1_score, cohen_kappa_score,
                                 confusion_matrix)

    human = pd.read_csv(human_csv)
    key = pd.read_csv(key_csv)
    base = pd.read_csv(baseline_csv)

    df = human.merge(key, on="id").merge(base, on="id", suffixes=("", "_b"))
    df = df[df["human_type"].isin(["I", "A", "M"])].copy()   # O 제외
    print(f"분석 대상: {len(df)}개 (O/빈칸 제외)")

    if len(df) == 0:
        print("  ⚠️ human_type 이 안 채워졌거나 I/A/M 없음. 라벨링 먼저.")
        return

    # 사람 라벨 → 코드북 매핑: I=ignorance, A=ambiguity, M=metacognitive
    type_map = {"I": "ignorance", "A": "ambiguity", "M": "metacognitive"}
    df["human_axis"] = df["human_type"].map(type_map)

    # ── 방법 1: ours (auto_type) ──
    print("\n══ ours (6축 auto_type) vs human ══")
    _report(df["human_axis"], df["auto_type"])

    # ── 방법 2: evidential u → 유형 분리 ──
    # u 는 스칼라라 직접 I/A/M 못 냄. "u 가 유형을 가르나"를 본다:
    #   I형(증거부족)이 M형(판정실패)보다 u 가 높아야 evidential이 가른다는 것
    print("\n══ evidential u — 유형별 분포 (가르나 뭉개나) ══")
    for t in ["ignorance", "ambiguity", "metacognitive"]:
        m = df["human_axis"] == t
        if m.any():
            print(f"    {t:>14}: u mean={df.loc[m,'evidential_u'].mean():.4f} "
                  f"± {df.loc[m,'evidential_u'].std():.4f}  (n={m.sum()})")
    print("    → I형과 M형의 u 가 비슷하면 evidential은 둘을 뭉갬 (우리가 채우는 갭)")

    # ── 방법 3: MC dropout var → 유형 분리 ──
    print("\n══ MC dropout variance — 유형별 분포 ══")
    for t in ["ignorance", "ambiguity", "metacognitive"]:
        m = df["human_axis"] == t
        if m.any():
            print(f"    {t:>14}: var mean={df.loc[m,'mc_var'].mean():.6f} "
                  f"± {df.loc[m,'mc_var'].std():.6f}  (n={m.sum()})")

    print("\n핵심: ours 가 human 을 높은 κ/F1 로 복원하고,")
    print("      evidential u / MC var 는 I형과 M형을 못 가르면 → 우리 기여 확립.")


def _report(y_true, y_pred):
    from sklearn.metrics import (accuracy_score, f1_score, cohen_kappa_score,
                                 confusion_matrix)
    labels = ["ignorance", "ambiguity", "metacognitive"]
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    kappa = cohen_kappa_score(y_true, y_pred, labels=labels)
    print(f"    Accuracy={acc:.4f}  Macro-F1={f1:.4f}  Cohen κ={kappa:.4f}")
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    print(f"    confusion (행=human, 열=pred): {labels}")
    for i, l in enumerate(labels):
        print(f"      {l:>14}: {cm[i].tolist()}")


# ── 실행 (새 셀에서 순서대로) ────────────────────────────────────────────────
# 1) evidential 재학습 (~9분):
#    train_evidential(seed=42)
#
# 2) baseline 점수 추출 (evidential u + MC dropout):
#    extract_baseline_scores()
#
# 3) (사람 라벨링 끝난 뒤) 분석:
#    analyze_vs_human()

print("Phase 1 baseline 모듈 로드됨.")
print("실행: train_evidential(seed=42) → extract_baseline_scores() → (라벨 후) analyze_vs_human()")

Phase 1 baseline 모듈 로드됨.
실행: train_evidential(seed=42) → extract_baseline_scores() → (라벨 후) analyze_vs_human()


In [12]:
train_evidential(seed=42, ckpt_out="/kaggle/input/datasets/terryterry9/evidential-baseline/evidential_seed42.pt")

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/tmp/ipykernel_58/1374529453.py:134: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda"))
/tmp/ipykernel_58/1374529453.py:144: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


  evidential epoch 1: loss=0.7321 acc=0.4204
  evidential epoch 2: loss=0.7410 acc=0.4990
  evidential epoch 3: loss=0.7372 acc=0.4990


/tmp/ipykernel_58/1374529453.py:167: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


  evidential val acc = 0.4738  (우리 모델 ~0.63과 비교)
  saved → /kaggle/working/evidential_seed42.pt


EvidentialBERT(
  (bert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): MultiHeadSelfAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
            (lin1): Linear(i

In [6]:
extract_baseline_scores(
    our_ckpt_fmt="/kaggle/input/datasets/terryterry9/epistemicbert-3seed/epistemic_bert_seed{}.pt"
)

/tmp/ipykernel_58/1374529453.py:268: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_58/1374529453.py:284: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_58/1374529453.py:202: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


  seed 42: baseline 점수 추출 완료 (누적 63개)
  seed 7: baseline 점수 추출 완료 (누적 107개)
  seed 123: baseline 점수 추출 완료 (누적 219개)

baseline 점수 → /kaggle/working/baseline_scores.csv  (219개)

  evidential u (confident error): mean=0.2034  std=0.0942  min=0.1137  max=0.5963
  → u 가 낮고 분산 작으면: evidential도 confident error를 '확신'으로 봄
     (= 유형 구분 불가 가능성. 사람 라벨과 합쳐서 최종 확인)


'/kaggle/working/baseline_scores.csv'

In [7]:
analyze_vs_human(
    human_csv="/kaggle/input/datasets/terryterry9/relabel-final/relabel_final_for_analysis (1).csv",
    key_csv="/kaggle/input/datasets/terryterry9/epistemicbert-3seed/cw_answer_key.csv",
    baseline_csv="/kaggle/input/datasets/terryterry9/relabel-final/baseline_scores.csv"
)

분석 대상: 219개 (O/빈칸 제외)

══ ours (6축 auto_type) vs human ══
    Accuracy=0.3836  Macro-F1=0.2458  Cohen κ=-0.0596
    confusion (행=human, 열=pred): ['ignorance', 'ambiguity', 'metacognitive']
           ignorance: [79, 23, 4]
           ambiguity: [32, 4, 0]
       metacognitive: [27, 11, 1]

══ evidential u — 유형별 분포 (가르나 뭉개나) ══
         ignorance: u mean=0.1796 ± 0.0728  (n=123)
         ambiguity: u mean=0.2042 ± 0.0847  (n=45)
     metacognitive: u mean=0.2601 ± 0.1222  (n=51)
    → I형과 M형의 u 가 비슷하면 evidential은 둘을 뭉갬 (우리가 채우는 갭)

══ MC dropout variance — 유형별 분포 ══
         ignorance: var mean=0.016824 ± 0.040809  (n=123)
         ambiguity: var mean=0.007267 ± 0.020282  (n=45)
     metacognitive: var mean=0.028018 ± 0.052126  (n=51)

핵심: ours 가 human 을 높은 κ/F1 로 복원하고,
      evidential u / MC var 는 I형과 M형을 못 가르면 → 우리 기여 확립.


In [7]:
def boot(y, s, n=2000):
    a=[]; idx=np.arange(len(y))
    for _ in range(n):
        b=np.random.choice(idx,len(idx),replace=True)
        if len(np.unique(y[b]))<2: continue
        a.append(roc_auc_score(y[b],s[b]))
    a=np.array(a); return a.mean(), np.percentile(a,2.5), np.percentile(a,97.5)

print("=== ignorance가 sufficiency(I vs 충분)를 가르나 — 분할 비교 ===\n")
for g in ['REFUTES','SUPPORTS','NEI']:
    sub = m[m['gold']==g].copy()
    print(f"── gold={g} ──")
    # 분할A: I vs M
    aA = sub[sub['human_type'].isin(['I','M'])]
    yA=(aA['human_type']=='I').astype(int).values
    if yA.sum()>=2 and (1-yA).sum()>=2:
        mn,lo,hi=boot(yA, aA['ignorance'].values)
        print(f"  A [I vs M]       n={yA.sum()}/{(1-yA).sum():>2}  AUROC={mn:.3f} [{lo:.3f},{hi:.3f}]")
    # 분할B: I vs (A+M) = sufficiency 축
    yB=(sub['human_type']=='I').astype(int).values
    if yB.sum()>=2 and (1-yB).sum()>=2:
        mn,lo,hi=boot(yB, sub['ignorance'].values)
        print(f"  B [suff:I vs AM] n={yB.sum()}/{(1-yB).sum():>2}  AUROC={mn:.3f} [{lo:.3f},{hi:.3f}]")
    print()

# raw 분포 — 가장 정직한 그림
print("=== ignorance raw 분포 (작은 셀이 정말 갈리나, 눈으로) ===")
for g in ['NEI','SUPPORTS']:
    print(f"\ngold={g}:")
    sub = m[m['gold']==g]
    for t in ['I','A','M']:
        vals = np.sort(sub[sub['human_type']==t]['ignorance'].values).round(3)
        print(f"  {t} (n={len(vals):>2}): {vals}")

=== ignorance가 sufficiency(I vs 충분)를 가르나 — 분할 비교 ===

── gold=REFUTES ──
  A [I vs M]       n=83/30  AUROC=0.517 [0.382,0.657]
  B [suff:I vs AM] n=83/48  AUROC=0.475 [0.366,0.585]

── gold=SUPPORTS ──
  A [I vs M]       n=7/16  AUROC=0.779 [0.527,0.967]
  B [suff:I vs AM] n=7/22  AUROC=0.775 [0.524,0.975]

── gold=NEI ──
  A [I vs M]       n=33/ 5  AUROC=0.812 [0.588,1.000]
  B [suff:I vs AM] n=33/26  AUROC=0.601 [0.447,0.749]

=== ignorance raw 분포 (작은 셀이 정말 갈리나, 눈으로) ===

gold=NEI:
  I (n=33): [0.113 0.134 0.141 0.198 0.205 0.21  0.267 0.274 0.277 0.293 0.329 0.336
 0.357 0.375 0.395 0.4   0.411 0.44  0.45  0.458 0.463 0.504 0.52  0.521
 0.53  0.542 0.564 0.573 0.603 0.642 0.652 0.666 0.754]
  A (n=21): [0.13  0.136 0.172 0.177 0.202 0.202 0.222 0.24  0.319 0.38  0.38  0.395
 0.492 0.508 0.515 0.537 0.565 0.568 0.583 0.602 0.66 ]
  M (n= 5): [0.085 0.103 0.207 0.293 0.412]

gold=SUPPORTS:
  I (n= 7): [0.347 0.399 0.584 0.604 0.605 0.669 0.804]
  A (n= 6): [0.221 0.408 0.48  0.507 0.5

In [ ]:
# ── 이전 셀에서 epistemic_bert.py 를 먼저 실행해야 함 ──────────────────────
# (CFG, EpistemicBERT, EpistemicFieldClassifier, failure_typing 등이 이미 정의됨)

"""
Phase 2 — climate_fever 로더 + VitaminC-trained 모델 전이 평가.

allenai/scifact: 로딩 스크립트 방식 deprecated → 막힘
BeIR/scifact:   corpus+queries만 있고 SUPPORT/CONTRADICT 라벨 없음 → 우리 용도 불가

climate_fever 선택 이유:
  - pre-paired (claim, evidence, label), 검색 불필요
  - SUPPORTS / REFUTES / NOT_ENOUGH_INFO 라벨 완비 (FEVER와 동일 구조)
  - 기후과학 도메인 → VitaminC(Wikipedia)와 도메인 완전 달라서
    "다른 fact-verification 도메인에서도 진단 구조 성립하나"를 더 강하게 테스트

평가 전략: VitaminC-trained 모델(seed 42)을 climate_fever에 그대로 적용.
  재학습 없이 "SUPPORTS→truth / REFUTES→error / NEI→ignorance 매핑이
  다른 도메인 fact-verification에서도 재현되나" 확인.
"""

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import DistilBertTokenizerFast
from datasets import load_dataset


# ─── 라벨 매핑 ────────────────────────────────────────────────────────────────
CF_LABEL_MAP = {
    "SUPPORTS":          0,
    "REFUTES":           1,
    "NOT_ENOUGH_INFO":   2,
    "DISPUTED":          2,   # DISPUTED는 판정 불가 → NEI 취급
}


# ─── 1. 진단 ──────────────────────────────────────────────────────────────────
def diagnose_climate_fever():
    """climate_fever 구조 출력. 처음 한 번 실행해서 field명 확인."""
    print("Loading climate_fever ...")
    try:
        ds = load_dataset("climate_fever")
    except Exception as e:
        print(f"  ✗ 로드 실패: {e}")
        return None

    print(f"\n  splits: {list(ds.keys())}")
    for split_name, split in ds.items():
        print(f"\n  [{split_name}] {len(split)} rows")
        print(f"    features: {list(split.features.keys())}")
        if len(split) > 0:
            ex = split[0]
            print(f"    첫 번째 예시:")
            for k, v in ex.items():
                v_repr = repr(v)[:200] + ("..." if len(repr(v)) > 200 else "")
                print(f"      {k}: {v_repr}")

    # 라벨 분포 확인
    for split_name, split in ds.items():
        from collections import Counter
        labels = [row.get("claim_label", row.get("label", "?")) for row in split]
        dist = Counter(labels)
        print(f"\n  [{split_name}] 라벨 분포: {dict(dist)}")

    return ds


# ─── 2. (claim, evidence_text, label) 트리플 구성 ────────────────────────────
def build_climate_fever_triples(split, max_evidence_len=400):
    """climate_fever split → [(claim, evidence_text, label_int)] 리스트.

    evidence 구조가 두 가지 버전에 방어적으로 대응:
      A) evidences: [{"evidence": str, "evidence_label": str, ...}, ...]
      B) evidences: [str, str, ...]   (단순 텍스트 리스트)
      C) evidence: str                (단일 문자열)
    """
    triples = []
    skipped = 0
    dropped_disputed = 0
    for row in split:
        claim = row.get("claim", "") or ""
        if not claim.strip():
            skipped += 1
            continue

        # 라벨 결정 — climate_fever는 정수 라벨:
        #   0=SUPPORTS, 1=REFUTES, 2=NOT_ENOUGH_INFO, 3=DISPUTED
        raw_label = row.get("claim_label", row.get("label", 2))
        if isinstance(raw_label, int):
            if raw_label == 3:                 # DISPUTED 제외 (우리 모델 미학습 라벨)
                dropped_disputed += 1
                continue
            label = raw_label if raw_label in (0, 1, 2) else 2
        else:                                  # 문자열 버전 대비
            if raw_label == "DISPUTED":
                dropped_disputed += 1
                continue
            label = CF_LABEL_MAP.get(raw_label, 2)

        # evidence 텍스트 추출 (evidence_label 정수는 무시, 텍스트만 사용)
        ev = row.get("evidences", row.get("evidence", None))
        if ev is None:
            ev_text = ""
        elif isinstance(ev, str):
            ev_text = ev
        elif isinstance(ev, list) and ev:
            first = ev[0]
            if isinstance(first, dict):
                ev_text = first.get("evidence", first.get("text", ""))
            elif isinstance(first, str):
                ev_text = first
            else:
                ev_text = str(first)
        else:
            ev_text = ""

        ev_text = ev_text[:max_evidence_len]
        triples.append((claim, ev_text, label))

    print(f"  트리플 생성: {len(triples)}개 (skipped {skipped}, DISPUTED 제외 {dropped_disputed})")
    from collections import Counter
    dist = Counter(t[2] for t in triples)
    LNAMES = {0: "SUPPORTS", 1: "REFUTES", 2: "NEI"}
    print(f"  라벨 분포: " + "  ".join(f"{LNAMES[l]}={dist[l]}" for l in [0, 1, 2]))
    return triples


# ─── 3. ClimateFEVERDataset ───────────────────────────────────────────────────
class ClimateFEVERDataset(Dataset):
    """FEVERDataset과 동일한 인터페이스 — input_ids/attention_mask/label/idx 반환."""

    def __init__(self, triples, tokenizer, max_length):
        self.triples = triples
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.triples)

    def __getitem__(self, idx):
        claim, evidence, label = self.triples[idx]
        enc = self.tokenizer(
            claim, evidence,
            max_length=self.max_length, padding="max_length",
            truncation=True, return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label":          torch.tensor(label, dtype=torch.long),
            "idx":            torch.tensor(idx,  dtype=torch.long),
        }


# ─── 4. 로드 ─────────────────────────────────────────────────────────────────
def load_climate_fever(tokenizer, max_length, val_size=None):
    """climate_fever 로드 → (val_loader, triples).

    climate_fever는 test split만 라벨 있음 → 이걸 eval 용도로 사용.
    (train split이 없거나 라벨 없으면 test를 9:1 분할해서 진단용 val 구성)
    """
    print("Loading climate_fever ...")
    ds = load_dataset("climate_fever")

    # 라벨 있는 split 찾기
    labeled_split = None
    for name in ["test", "validation", "train"]:
        if name in ds:
            sample = ds[name][0]
            if "claim_label" in sample or "label" in sample:
                labeled_split = ds[name]
                print(f"  사용 split: '{name}' ({len(labeled_split)}개)")
                break

    if labeled_split is None:
        labeled_split = list(ds.values())[0]
        print(f"  사용 split: 첫 번째 ({len(labeled_split)}개)")

    triples = build_climate_fever_triples(labeled_split)

    # val 구성: 전체 사용 (fine-tune 없이 전이 평가만 하므로 모두 val)
    n_val = val_size or len(triples)
    val_triples = triples[:n_val]

    val_ds = ClimateFEVERDataset(val_triples, tokenizer, max_length)
    val_loader = DataLoader(val_ds, batch_size=CFG["train"]["batch_size"],
                            shuffle=False, num_workers=0, pin_memory=True)
    print(f"  val loader: {len(val_triples)}개")
    return val_loader, val_triples


# ─── 5. 전이 평가 ────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate_transfer(model, val_loader, tokenizer, device):
    """VitaminC-trained 모델을 climate_fever에 그대로 적용.

    확인 포인트:
      ① SUPPORTS→truth / REFUTES→error / NEI→ignorance 매핑 재현
      ② identifiability (독립성) 유지
      ③ failure_typing의 ignorance형 지배 재현
    """
    model.eval()
    AXES  = EpistemicFieldClassifier.AXES
    LNAME = {0: "SUPPORTS", 1: "REFUTES", 2: "NEI"}

    # ① 축 ↔ 라벨 매핑 ────────────────────────────────────────────────────
    print("\n══════════════════════════════════════════════════════════")
    print("① climate_fever Axis ↔ Label Mapping (전이, 재학습 없음)")
    print("══════════════════════════════════════════════════════════")
    fields, labels = [], []
    for batch in val_loader:
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        with torch.cuda.amp.autocast():
            _, eout, _ = model(ids, mask)
        fields.append(eout["field"].float().cpu())
        labels.append(batch["label"])
    fld = torch.cat(fields).numpy()
    lab = torch.cat(labels).numpy()

    means = {}
    print(f"\n{'axis':>14}" + "".join(f"{LNAME[l]:>11}" for l in [0,1,2]))
    print("─" * 50)
    for i, ax in enumerate(AXES):
        row = ""
        for l in [0,1,2]:
            m = lab == l
            if m.any():
                v = fld[m, i].mean()
                means.setdefault(l, {})[ax] = v
                row += f"{v:>11.4f}"
            else:
                row += f"{'—':>11}"
        print(f"{ax:>14}{row}")

    def hi(axis_i, target):
        vals = {l: means[l].get(AXES[axis_i], 0) for l in means}
        if not vals: return "데이터 없음"
        top = max(vals, key=vals.get)
        mark = "✓" if top == target else "✗"
        return f"{mark} (top={LNAME[top]} {vals[top]:.3f})"

    print(f"\n  매핑 점검:")
    print(f"    truth     → SUPPORTS  {hi(0, 0)}")
    print(f"    error     → REFUTES   {hi(1, 1)}")
    print(f"    ignorance → NEI       {hi(5, 2)}")

    # VitaminC vs climate_fever 매핑 비교 요약
    print(f"\n  ※ VitaminC 결과: truth→SUP 0.623  error→REF 0.638  ign→NEI 0.485")
    print(f"    위 숫자와 비교 — 다른 도메인에서도 같은 패턴이 나오면 전이 성공")

    # ② identifiability ────────────────────────────────────────────────────
    print("\n══════════════════════════════════════════════════════════")
    print("② Identifiability (climate_fever)")
    print("══════════════════════════════════════════════════════════")
    identifiability_probe(model, val_loader, device)

    # ③ failure typing ─────────────────────────────────────────────────────
    print("\n══════════════════════════════════════════════════════════")
    print("③ Failure Typing (climate_fever)")
    print("══════════════════════════════════════════════════════════")
    failure_typing(model, val_loader, tokenizer, device)


# ─── 6. Main ─────────────────────────────────────────────────────────────────
def main(mode="diagnose",
         ckpt="/kaggle/working/epistemic_bert_seed42.pt"):
    """
    mode:
      "diagnose"  — climate_fever 구조 출력 (먼저 실행)
      "transfer"  — VitaminC 모델을 climate_fever에 그대로 평가 (재학습 없음)
    """
    if mode == "diagnose":
        diagnose_climate_fever()
        return

    device = CFG["device"]
    tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
    val_loader, _ = load_climate_fever(tokenizer, CFG["train"]["max_length"])

    model = EpistemicBERT(**CFG["model"]).to(device)
    model.token_nov.set_special_tokens(tokenizer.all_special_ids)
    model.attn_ign.sep_id = tokenizer.sep_token_id
    print(f"\nLoading checkpoint: {ckpt}")
    model.load_state_dict(torch.load(ckpt, map_location=device))
    model.eval()

    evaluate_transfer(model, val_loader, tokenizer, device)


# ── 실행 (새 셀에서 원하는 줄만) ─────────────────────────────────────────────
# 1) 구조 확인 먼저:
#    main("diagnose")
#
# 2) 전이 평가:
#    main("transfer", ckpt="/kaggle/working/epistemic_bert_seed42.pt")

main("diagnose")   # ← 처음엔 이것만